In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week8-assignment-4"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

## Understanding Rank, Dense Rank & Row Number

In [2]:
## /public/trendytech/datasets/windowdatamodified.csv

In [3]:
windowDF = spark.read \
.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/public/trendytech/datasets/windowdatamodified.csv")

In [4]:
windowDF.sort("country").show(10)

+---------------+-------+-----------+-------------+------------+
|        country|weeknum|numinvoices|totalquantity|invoicevalue|
+---------------+-------+-----------+-------------+------------+
|      Australia|     49|          1|          214|       258.9|
|      Australia|     48|          1|          107|      358.25|
|      Australia|     50|          2|          133|      387.95|
|        Austria|     50|          2|            3|      257.04|
|        Bahrain|     51|          1|           54|      205.74|
|        Belgium|     48|          1|          528|       800.0|
|        Belgium|     50|          2|          285|      625.16|
|        Belgium|     51|          2|          942|       800.0|
|Channel Islands|     49|          1|           80|      363.53|
|         Cyprus|     50|          1|          917|     1590.82|
+---------------+-------+-----------+-------------+------------+
only showing top 10 rows



In [5]:
from pyspark.sql import Window

#### running total

In [6]:
mywindow = Window.partitionBy("country").orderBy("weeknum").rowsBetween(Window.unboundedPreceding,Window.currentRow)

In [7]:
resdf = windowDF.withColumn("running_tot",sum("invoicevalue").over(mywindow))

In [8]:
resdf.show(10)

+-------+-------+-----------+-------------+------------+------------------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|       running_tot|
+-------+-------+-----------+-------------+------------+------------------+
| Sweden|     50|          3|         3714|      2646.3|            2646.3|
|Germany|     48|         11|         1795|      1600.0|            1600.0|
|Germany|     49|         12|         1852|      1800.0|            3400.0|
|Germany|     50|         15|         1973|      1800.0|            5200.0|
|Germany|     51|          5|         1103|      1600.0|            6800.0|
| France|     48|          4|         1299|       500.0|             500.0|
| France|     49|          9|         2303|       500.0|            1000.0|
| France|     50|          6|          529|      537.32|1537.3200000000002|
| France|     51|          5|          847|       500.0|2037.3200000000002|
|Belgium|     48|          1|          528|       800.0|             800.0|
+-------+---

### defining window with out rows between

In [9]:
window1 = Window.partitionBy("country").orderBy(desc("invoicevalue"))   ## this window.orderBy() is different from df.sortBy()

###  rank().over()

In [10]:
newdf = windowDF.withColumn("rank",rank().over(window1))

In [11]:
newdf.show(20)

+-------+-------+-----------+-------------+------------+----+
|country|weeknum|numinvoices|totalquantity|invoicevalue|rank|
+-------+-------+-----------+-------------+------------+----+
| Sweden|     50|          3|         3714|      2646.3|   1|
|Germany|     49|         12|         1852|      1800.0|   1|
|Germany|     50|         15|         1973|      1800.0|   1|
|Germany|     48|         11|         1795|      1600.0|   3|
|Germany|     51|          5|         1103|      1600.0|   3|
| France|     50|          6|          529|      537.32|   1|
| France|     51|          5|          847|       500.0|   2|
| France|     49|          9|         2303|       500.0|   2|
| France|     48|          4|         1299|       500.0|   2|
|Belgium|     48|          1|          528|       800.0|   1|
|Belgium|     51|          2|          942|       800.0|   1|
|Belgium|     50|          2|          285|      625.16|   3|
|Finland|     50|          1|         1254|       892.8|   1|
|  India

###  denserank().over()

In [12]:
denserank = windowDF.withColumn("denserank",dense_rank().over(window1))

In [13]:
denserank.show()

+-------+-------+-----------+-------------+------------+---------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|denserank|
+-------+-------+-----------+-------------+------------+---------+
| Sweden|     50|          3|         3714|      2646.3|        1|
|Germany|     49|         12|         1852|      1800.0|        1|
|Germany|     50|         15|         1973|      1800.0|        1|
|Germany|     48|         11|         1795|      1600.0|        2|
|Germany|     51|          5|         1103|      1600.0|        2|
| France|     50|          6|          529|      537.32|        1|
| France|     51|          5|          847|       500.0|        2|
| France|     49|          9|         2303|       500.0|        2|
| France|     48|          4|         1299|       500.0|        2|
|Belgium|     48|          1|          528|       800.0|        1|
|Belgium|     51|          2|          942|       800.0|        1|
|Belgium|     50|          2|          285|      625.16|      

###  rownum().over()

In [17]:
rownumDF = windowDF.withColumn("rownum",row_number().over(window1))

In [18]:
rownumDF.show()

+-------+-------+-----------+-------------+------------+------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|rownum|
+-------+-------+-----------+-------------+------------+------+
| Sweden|     50|          3|         3714|      2646.3|     1|
|Germany|     49|         12|         1852|      1800.0|     1|
|Germany|     50|         15|         1973|      1800.0|     2|
|Germany|     48|         11|         1795|      1600.0|     3|
|Germany|     51|          5|         1103|      1600.0|     4|
| France|     50|          6|          529|      537.32|     1|
| France|     51|          5|          847|       500.0|     2|
| France|     49|          9|         2303|       500.0|     3|
| France|     48|          4|         1299|       500.0|     4|
|Belgium|     48|          1|          528|       800.0|     1|
|Belgium|     51|          2|          942|       800.0|     2|
|Belgium|     50|          2|          285|      625.16|     3|
|Finland|     50|          1|         12

In [19]:
rownumDF.where("rownum = 1").show()  ##top record from each country

+---------------+-------+-----------+-------------+------------+------+
|        country|weeknum|numinvoices|totalquantity|invoicevalue|rownum|
+---------------+-------+-----------+-------------+------------+------+
|         Sweden|     50|          3|         3714|      2646.3|     1|
|        Germany|     49|         12|         1852|      1800.0|     1|
|         France|     50|          6|          529|      537.32|     1|
|        Belgium|     48|          1|          528|       800.0|     1|
|        Finland|     50|          1|         1254|       892.8|     1|
|          India|     49|          5|         1280|      3284.1|     1|
|          Italy|     48|          1|          164|       427.8|     1|
|      Lithuania|     48|          3|          622|     1598.06|     1|
|         Norway|     48|          1|         1852|     1919.14|     1|
|          Spain|     50|          2|          400|     1049.01|     1|
|        Denmark|     49|          1|          454|      1281.5|

In [20]:
rownumDF.where("rownum < 4").show()  ##top 3  from each country

+---------+-------+-----------+-------------+------------+------+
|  country|weeknum|numinvoices|totalquantity|invoicevalue|rownum|
+---------+-------+-----------+-------------+------------+------+
|   Sweden|     50|          3|         3714|      2646.3|     1|
|  Germany|     49|         12|         1852|      1800.0|     1|
|  Germany|     50|         15|         1973|      1800.0|     2|
|  Germany|     48|         11|         1795|      1600.0|     3|
|   France|     50|          6|          529|      537.32|     1|
|   France|     51|          5|          847|       500.0|     2|
|   France|     49|          9|         2303|       500.0|     3|
|  Belgium|     48|          1|          528|       800.0|     1|
|  Belgium|     51|          2|          942|       800.0|     2|
|  Belgium|     50|          2|          285|      625.16|     3|
|  Finland|     50|          1|         1254|       892.8|     1|
|    India|     49|          5|         1280|      3284.1|     1|
|    India